In [1]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [2]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-small.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-small")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-small
Contents: ['snapshots', 'blobs', 'refs']


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import nltk

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [4]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df_raw = pd.read_csv('ExioNAICS.csv')

naics_corpus = df_raw[['NAICS Code', 'NAICS Title', 'Description']].drop_duplicates(subset='NAICS Code').copy()
naics_corpus['NAICS Code'] = naics_corpus['NAICS Code'].astype(str)
naics_corpus['clean_title'] = naics_corpus['NAICS Title'].apply(preprocess_text)
naics_corpus['clean_desc'] = naics_corpus['Description'].apply(preprocess_text)
naics_corpus['corpus_text'] = naics_corpus['clean_title'] + ' ' + naics_corpus['clean_desc']
naics_corpus = naics_corpus.reset_index(drop=True)

code_to_idx = {code: i for i, code in enumerate(naics_corpus['NAICS Code'])}
idx_to_code = {i: code for code, i in code_to_idx.items()}

print(f"NAICS corpus: {len(naics_corpus)} unique codes")
print(f"Sample corpus entry:")
print(f"  Code: {naics_corpus['NAICS Code'].iloc[0]}")
print(f"  Title: {naics_corpus['NAICS Title'].iloc[0]}")
print(f"  Corpus text (first 200 chars): {naics_corpus['corpus_text'].iloc[0][:200]}")

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics_idx'] = df['NAICS Code'].map(code_to_idx)

missing = df['naics_idx'].isna().sum()
if missing > 0:
    print(f"WARNING: {missing} rows have no matching NAICS code in corpus, dropping them")
    df = df.dropna(subset=['naics_idx']).reset_index(drop=True)
df['naics_idx'] = df['naics_idx'].astype(int)

print(f"\nDataset: {len(df)} samples, {df['naics_idx'].nunique()} unique NAICS codes")

NAICS corpus: 1115 unique codes
Sample corpus entry:
  Code: 448150
  Title: Clothing Accessories Stores
  Corpus text (first 200 chars): clothing accessories stores industry comprises establishments primarily engaged retailing general specialized lines new clothing clothing accessories hats caps costume jewelry gloves handbags ties wig

Dataset: 20535 samples, 1115 unique NAICS codes


In [5]:
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 128
BATCH_SIZE = 64
NUM_EPOCHS = 30
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
TEMPERATURE = 0.05
SEED = 42
VAL_RATIO = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)

print("=== Contrastive Learning Config ===")
print(f"  Model:         {MODEL_NAME}")
print(f"  Max length:    {MAX_LENGTH}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Epochs:        {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Temperature:   {TEMPERATURE}")
print(f"  Val ratio:     {VAL_RATIO}")

=== Contrastive Learning Config ===
  Model:         microsoft/deberta-v3-small
  Max length:    128
  Batch size:    64
  Epochs:        30
  Learning rate: 2e-05
  Temperature:   0.05
  Val ratio:     0.1


In [6]:
class DeBERTaEncoder(nn.Module):
    """DeBERTa-v3 with mean pooling to produce sentence embeddings."""

    def __init__(self, model_name, normalize=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.normalize = normalize
        self.hidden_size = self.backbone.config.hidden_size

    def mean_pool(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.mean_pool(outputs.last_hidden_state, attention_mask)
        if self.normalize:
            embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DeBERTaEncoder(MODEL_NAME).to(device)

print(f"Encoder hidden size: {model.hidden_size}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

test_input = tokenizer("test sentence", return_tensors="pt", max_length=MAX_LENGTH, truncation=True, padding=True)
test_input = {k: v.to(device) for k, v in test_input.items() if k in ['input_ids', 'attention_mask']}
with torch.no_grad():
    test_emb = model(**test_input)
print(f"Output embedding shape: {test_emb.shape}")
print(f"Embedding norm: {test_emb.norm().item():.4f} (should be ~1.0)")
print(f"Any NaN: {torch.isnan(test_emb).any().item()}")

The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder hidden size: 768
Total params: 141,304,320
Output embedding shape: torch.Size([1, 768])
Embedding norm: 1.0000 (should be ~1.0)
Any NaN: False


In [7]:
def mnr_loss(query_embeddings, doc_embeddings, temperature=0.05):
    """Multiple Negatives Ranking (MNR) / InfoNCE loss.

    For each query i, doc i is the positive. All other docs in the batch are negatives.
    """
    similarity = torch.matmul(query_embeddings, doc_embeddings.T) / temperature
    labels = torch.arange(similarity.size(0), device=similarity.device)
    return F.cross_entropy(similarity, labels)

print("MNR loss function defined.")
print("  - Each query's matching NAICS doc is the positive")
print("  - All other docs in the batch serve as negatives")
print(f"  - Temperature: {TEMPERATURE}")

MNR loss function defined.
  - Each query's matching NAICS doc is the positive
  - All other docs in the batch serve as negatives
  - Temperature: 0.05


In [8]:
class ContrastiveDataset(Dataset):
    """Each sample returns a (query, positive_doc) pair.
    Query = company description, Positive doc = matching NAICS corpus text.
    """

    def __init__(self, descriptions, naics_indices, corpus_texts, tokenizer, max_length):
        self.descriptions = descriptions
        self.naics_indices = naics_indices
        self.corpus_texts = corpus_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        query = self.descriptions[idx]
        doc = self.corpus_texts[self.naics_indices[idx]]

        query_enc = self.tokenizer(
            query, max_length=self.max_length, truncation=True,
            padding='max_length', return_tensors='pt'
        )
        doc_enc = self.tokenizer(
            doc, max_length=self.max_length, truncation=True,
            padding='max_length', return_tensors='pt'
        )

        return {
            'query_input_ids': query_enc['input_ids'].squeeze(0),
            'query_attention_mask': query_enc['attention_mask'].squeeze(0),
            'doc_input_ids': doc_enc['input_ids'].squeeze(0),
            'doc_attention_mask': doc_enc['attention_mask'].squeeze(0),
            'naics_idx': self.naics_indices[idx],
        }

print("ContrastiveDataset defined.")

ContrastiveDataset defined.


In [9]:
corpus_texts = naics_corpus['corpus_text'].tolist()

label_counts = df['naics_idx'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df = df[df['naics_idx'].isin(valid_labels)].reset_index(drop=True)
print(f"After filtering singletons: {len(df)} samples, {df['naics_idx'].nunique()} codes")

train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=VAL_RATIO, random_state=SEED, stratify=df['naics_idx']
)

train_dataset = ContrastiveDataset(
    df['clean_description'].iloc[train_idx].tolist(),
    df['naics_idx'].iloc[train_idx].tolist(),
    corpus_texts, tokenizer, MAX_LENGTH
)
val_dataset = ContrastiveDataset(
    df['clean_description'].iloc[val_idx].tolist(),
    df['naics_idx'].iloc[val_idx].tolist(),
    corpus_texts, tokenizer, MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

After filtering singletons: 20533 samples, 1113 codes
Train: 18479 samples, 289 batches
Val:   2054 samples, 33 batches


In [10]:
@torch.no_grad()
def encode_corpus(model, corpus_texts, tokenizer, max_length, batch_size=128):
    """Pre-encode all NAICS corpus entries into embeddings."""
    model.eval()
    all_embeddings = []
    for i in range(0, len(corpus_texts), batch_size):
        batch_texts = corpus_texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts, max_length=max_length, truncation=True,
            padding=True, return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
        emb = model(**enc)
        all_embeddings.append(emb.cpu())
    return torch.cat(all_embeddings, dim=0)


@torch.no_grad()
def evaluate(model, val_loader, corpus_embeddings, temperature):
    """Evaluate Top-k retrieval accuracy on validation set."""
    model.eval()
    all_query_embs = []
    all_labels = []

    for batch in val_loader:
        query_emb = model(
            batch['query_input_ids'].to(device),
            batch['query_attention_mask'].to(device),
        )
        all_query_embs.append(query_emb.cpu())
        all_labels.append(batch['naics_idx'])

    query_embs = torch.cat(all_query_embs, dim=0)
    labels = torch.cat(all_labels, dim=0)

    similarity = torch.matmul(query_embs, corpus_embeddings.T)

    top1, top5, top10 = 0, 0, 0
    n = len(labels)
    for i in range(n):
        topk_indices = similarity[i].topk(10).indices
        true_label = labels[i].item()
        if true_label == topk_indices[0].item():
            top1 += 1
        if true_label in topk_indices[:5]:
            top5 += 1
        if true_label in topk_indices[:10]:
            top10 += 1

    loss_sum = 0.0
    sim_batched = torch.matmul(query_embs, corpus_embeddings.T) / temperature
    for i in range(n):
        loss_sum += F.cross_entropy(sim_batched[i].unsqueeze(0), labels[i].unsqueeze(0)).item()

    return {
        'val_loss': loss_sum / n,
        'top1_accuracy': top1 / n,
        'top5_accuracy': top5 / n,
        'top10_accuracy': top10 / n,
    }

print("Evaluation function defined: retrieval over full NAICS corpus")

Evaluation function defined: retrieval over full NAICS corpus


In [11]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

os.makedirs("results", exist_ok=True)
best_top1 = 0.0
best_epoch = 0
patience = 7
patience_counter = 0
epoch_log = []

print(f"Optimizer: AdamW, lr={LEARNING_RATE}, wd={WEIGHT_DECAY}")
print(f"Scheduler: linear warmup ({warmup_steps} steps) + linear decay")
print(f"Total training steps: {total_steps}")
print(f"Early stopping patience: {patience} (on Top-1 accuracy)")
print(f"\n{'='*80}")
print(f"{'Epoch':>5} {'Train Loss':>11} {'Val Loss':>9} {'Top-1':>7} {'Top-5':>7} {'Top-10':>7} {'LR':>12}")
print(f"{'='*80}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        query_emb = model(
            batch['query_input_ids'].to(device),
            batch['query_attention_mask'].to(device),
        )
        doc_emb = model(
            batch['doc_input_ids'].to(device),
            batch['doc_attention_mask'].to(device),
        )

        loss = mnr_loss(query_emb, doc_emb, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_loss / num_batches
    current_lr = scheduler.get_last_lr()[0]

    corpus_embeddings = encode_corpus(model, corpus_texts, tokenizer, MAX_LENGTH)
    metrics = evaluate(model, val_loader, corpus_embeddings, TEMPERATURE)

    epoch_log.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'val_loss': metrics['val_loss'],
        'top1_accuracy': metrics['top1_accuracy'],
        'top5_accuracy': metrics['top5_accuracy'],
        'top10_accuracy': metrics['top10_accuracy'],
        'lr': current_lr,
    })

    print(f"{epoch:>5} {avg_train_loss:>11.4f} {metrics['val_loss']:>9.4f} "
          f"{metrics['top1_accuracy']:>7.4f} {metrics['top5_accuracy']:>7.4f} "
          f"{metrics['top10_accuracy']:>7.4f} {current_lr:>12.2e}")

    if metrics['top1_accuracy'] > best_top1:
        best_top1 = metrics['top1_accuracy']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "results/best_model.pt")
    else:
        patience_counter += 1

    pd.DataFrame(epoch_log).to_csv("results/contrastive_epoch_log.csv", index=False)

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break

print(f"\n{'='*80}")
print(f"Best Top-1: {best_top1:.4f} at epoch {best_epoch}")

model.load_state_dict(torch.load("results/best_model.pt", weights_only=True))
print("Loaded best model weights.")

Optimizer: AdamW, lr=2e-05, wd=0.01
Scheduler: linear warmup (867 steps) + linear decay
Total training steps: 8670
Early stopping patience: 7 (on Top-1 accuracy)

Epoch  Train Loss  Val Loss   Top-1   Top-5  Top-10           LR
    1      3.9473    6.1502  0.0112  0.0618  0.1008     6.67e-06
    2      2.8519    5.1323  0.0784  0.2298  0.3466     1.33e-05
    3      2.2697    4.7326  0.1217  0.3247  0.4499     2.00e-05
    4      1.9607    4.5613  0.1383  0.3729  0.4903     1.93e-05
    5      1.7482    4.4097  0.1553  0.3841  0.5253     1.85e-05
    6      1.5812    4.3337  0.1675  0.4036  0.5380     1.78e-05
    7      1.4452    4.3959  0.1655  0.4031  0.5282     1.70e-05
    8      1.3292    4.3757  0.1728  0.4080  0.5477     1.63e-05
    9      1.2348    4.3363  0.1723  0.4260  0.5526     1.56e-05
   10      1.1413    4.3998  0.1709  0.4216  0.5443     1.48e-05
   11      1.0524    4.4709  0.1650  0.4182  0.5424     1.41e-05
   12      0.9841    4.5561  0.1753  0.4124  0.5404     1

In [12]:
corpus_embeddings = encode_corpus(model, corpus_texts, tokenizer, MAX_LENGTH)
final_metrics = evaluate(model, val_loader, corpus_embeddings, TEMPERATURE)

print("=== Final Results (Best Model) ===")
print(f"  Top-1 Accuracy:  {final_metrics['top1_accuracy']:.4f}")
print(f"  Top-5 Accuracy:  {final_metrics['top5_accuracy']:.4f}")
print(f"  Top-10 Accuracy: {final_metrics['top10_accuracy']:.4f}")
print(f"  Val Loss:        {final_metrics['val_loss']:.4f}")
print(f"  Best Epoch:      {best_epoch}")

results = {
    "method": "DeBERTa-v3-small + MNR Contrastive Learning",
    "config": {
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "temperature": TEMPERATURE,
        "weight_decay": WEIGHT_DECAY,
        "num_epochs_trained": best_epoch,
    },
    "results": final_metrics,
}

with open("results/contrastive_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nResults saved to results/contrastive_results.json")

import zipfile, glob

with zipfile.ZipFile("results.zip", "w") as zf:
    for f in glob.glob("results/*.json") + glob.glob("results/*.csv") + glob.glob("results/*.pt"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results.zip")
print("\nDownloaded results.zip")

=== Final Results (Best Model) ===
  Top-1 Accuracy:  0.1753
  Top-5 Accuracy:  0.4124
  Top-10 Accuracy: 0.5404
  Val Loss:        4.5561
  Best Epoch:      12

Results saved to results/contrastive_results.json
  Added: results/contrastive_results.json
  Added: results/contrastive_epoch_log.csv
  Added: results/best_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded results.zip
